# BSH Activity Prediction: Non-Conserved Residue Embeddings

Instead of using the full protein embedding (which averages over all residues),
this model focuses on **variable/unique residue positions** — the positions that
differ between BSH enzymes and are most likely to determine substrate specificity.

**Approach:**
1. Use per-residue ProtT5 embeddings (seq_len × 1024)
2. Filter to non-conserved alignment positions (using MSA conservation scores)
3. Pool selected residue embeddings → fixed-size enzyme vector
4. Concatenate with amine Morgan fingerprint
5. Train RF, XGBoost, MLP with enzyme hold-out split

**Hypothesis:** Removing conserved (shared) residues and focusing on unique positions
should give the model a cleaner signal about what makes each enzyme different.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import h5py
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    balanced_accuracy_score
)
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    print("XGBoost not installed")
    HAS_XGB = False

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    HAS_RDKIT = True
except ImportError:
    print("RDKit not installed")
    HAS_RDKIT = False

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
MODEL_DIR = Path("../models")

# Model-specific output directories
MODEL_OUTPUT_DIR = OUTPUT_DIR / "model_outputs"
NONCONS_OUTPUT_DIR = MODEL_OUTPUT_DIR / "nonconserved_residue"
NONCONS_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
FP_BITS = 1024

## 1. Load Per-Residue Embeddings

In [ ]:
# Load per-residue ProtT5 embeddings
h5_path = DATA_DIR / "Seqs_list_total_per_residue.h5"

per_residue_embeddings = {}
with h5py.File(h5_path, 'r') as f:
    for key in f.keys():
        data = f[key][:]
        # Extract UniProt ID (last part after underscore)
        uniprot_id = key.split('_')[-1]
        per_residue_embeddings[uniprot_id] = data

print(f"Loaded per-residue embeddings for {len(per_residue_embeddings)} enzymes")
sample_key = list(per_residue_embeddings.keys())[0]
print(f"Example: {sample_key} -> shape {per_residue_embeddings[sample_key].shape}")

## 2. Load Conservation Scores & Map Alignment to Sequence Positions

In [ ]:
# Load conservation scores
df_cons = pd.read_csv(OUTPUT_DIR / "conservation_scores.csv")
print(f"Total alignment positions: {len(df_cons)}")

# Focus on core positions (not mostly gaps)
core_mask = df_cons['gap_fraction'] < 0.5
df_core = df_cons[core_mask].copy()
print(f"Core positions (gap < 50%): {len(df_core)}")

# Define conservation thresholds to try
THRESHOLDS = {
    'very_unique': 0.3,    # Most variable positions
    'unique': 0.5,         # Variable positions
    'non_conserved': 0.8,  # Moderately variable
    'all_variable': 0.95,  # Everything except highly conserved
}

for name, thresh in THRESHOLDS.items():
    n_pos = (df_core['conservation_score'] < thresh).sum()
    print(f"  {name} (< {thresh}): {n_pos} positions")

In [ ]:
# Parse alignment to map alignment positions -> sequence positions per enzyme
from Bio import SeqIO

alignment_path = OUTPUT_DIR / "bsh_aligned.fasta"

# Build mapping: enzyme_id -> {alignment_pos: sequence_pos}
alignment_to_seq = {}

for record in SeqIO.parse(alignment_path, 'fasta'):
    # Extract UniProt ID from record ID
    rec_id = record.id
    parts = rec_id.split('_')
    uniprot_id = parts[-1] if len(parts) > 1 else rec_id
    
    seq = str(record.seq)
    mapping = {}
    seq_pos = 0
    
    for aln_pos, char in enumerate(seq):
        if char != '-':  # Not a gap
            mapping[aln_pos] = seq_pos
            seq_pos += 1
    
    alignment_to_seq[uniprot_id] = mapping

print(f"Alignment mappings for {len(alignment_to_seq)} enzymes")

# Check overlap with per-residue embeddings
overlap = set(alignment_to_seq.keys()) & set(per_residue_embeddings.keys())
print(f"Enzymes with both alignment and embeddings: {len(overlap)}")

In [ ]:
def get_nonconserved_embedding(enzyme_id, conservation_threshold, pooling='mean'):
    """
    Extract and pool per-residue embeddings at non-conserved positions.
    
    Args:
        enzyme_id: UniProt ID
        conservation_threshold: Include positions with conservation < this value
        pooling: 'mean' or 'max'
    
    Returns:
        1024-dim pooled embedding vector, or None if not enough positions
    """
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    
    embed = per_residue_embeddings[enzyme_id]  # (seq_len, 1024)
    aln_map = alignment_to_seq[enzyme_id]       # {aln_pos: seq_pos}
    
    # Get non-conserved core alignment positions
    variable_aln_positions = df_core[
        df_core['conservation_score'] < conservation_threshold
    ]['alignment_position'].values
    
    # Map to sequence positions for this enzyme
    seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):  # Safety check
                seq_positions.append(seq_pos)
    
    if len(seq_positions) == 0:
        return None
    
    # Extract embeddings at selected positions and pool
    selected = embed[seq_positions]  # (n_selected, 1024)
    
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    
    return selected.mean(axis=0)


# Test it
test_enzyme = list(overlap)[0]
for name, thresh in THRESHOLDS.items():
    emb = get_nonconserved_embedding(test_enzyme, thresh)
    if emb is not None:
        print(f"{name} (< {thresh}): embedding shape = {emb.shape}")
    else:
        print(f"{name} (< {thresh}): no positions found")

## 3. Load Activity Labels & Amine Fingerprints

In [ ]:
# Load activity labels
df_activity = pd.read_csv(OUTPUT_DIR / "enzyme_amine_activity.csv")

controls = ['CTRL1', 'CTRL2', 'CTRL3', 'CTRL4', 'CTRL5', 'CTRL6', 'CTRL7']
canonical = ['taurine', 'glycine']
df_activity = df_activity[~df_activity['Enzyme'].isin(controls)]
df_activity = df_activity[~df_activity['Amine'].isin(canonical)]

# Aggregate by (Enzyme, Amine)
df_agg = df_activity.groupby(['Enzyme', 'Amine']).agg(
    active=('active_approach2', 'any'),
    n_products=('ProductName', 'count'),
    n_active_products=('active_approach2', 'sum')
).reset_index()

print(f"Aggregated data: {df_agg.shape}")
print(f"Class distribution: {df_agg['active'].value_counts().to_dict()}")
print(f"% Active: {100*df_agg['active'].mean():.1f}%")

In [ ]:
# Load amine SMILES and compute Morgan fingerprints
df_smiles = pd.read_excel(DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx")

name_map = {
    '2,3-Diaminopropinoic Acid': '2,3_diaminopropionic acid',
    '2-aminophenol': '2_aminophenol',
    '3-methoxytyramine HCl': '3_methoxytyramine',
    '4-aminophenol': '4_aminophenol',
    'L-Alanine': 'alanine',
    'L-Arginine': 'arginine',
    'Asparagine': 'asparagine',
    'Cadaverine': 'cadaverine',
    'L-Citrulline': 'citrulline',
    'L-Cysteine': 'cysteine',
    'Dopamine HCl': 'dopamine',
    'gamma-Aminobutyric acid >99%': 'gaba',
    'L-Glutamine': 'glutamine',
    'Glycyl-L-Valine': 'glyglycine',
    'L-Histidine': 'histidine',
    'L-Lysine': 'lysine',
    'L-Methionine': 'methionine',
    'L-Ornithine monohydrochloride': 'ornithine',
    'L-Phenylalanine': 'phenylalanine',
    'DL-Proline': 'proline',
    'Putrescine': 'putrescine',
    'L-Serine': 'serine',
    'L-Threonine': 'threonine',
    'Tryptamine': 'tryptamine',
}

amines_needed = df_agg['Amine'].unique()
amine_fingerprints = {}

if HAS_RDKIT:
    for _, row in df_smiles.iterrows():
        name = row['Compound_Name']
        smiles = row['SMILES']
        norm_name = name_map.get(name, name.lower().replace(' ', '_').replace('-', '_'))
        if pd.isna(smiles):
            continue
        smiles_clean = smiles.split('.')[0]
        mol = Chem.MolFromSmiles(smiles_clean)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=FP_BITS)
            amine_fingerprints[norm_name] = np.array(fp)

# Fill missing with zeros
for a in amines_needed:
    if a not in amine_fingerprints:
        amine_fingerprints[a] = np.zeros(FP_BITS, dtype=np.float32)

print(f"Amine fingerprints: {len(amine_fingerprints)}")

## 4. Build Feature Matrices for Each Conservation Threshold

In [ ]:
def build_feature_matrix(conservation_threshold, pooling='mean'):
    """
    Build X, y matrices using non-conserved residue embeddings.
    Returns X, y, enzyme_list, amine_list
    """
    X_list, y_list = [], []
    enzymes, amines = [], []
    skipped = 0
    
    for _, row in df_agg.iterrows():
        enzyme = row['Enzyme']
        amine = row['Amine']
        
        # Get non-conserved residue embedding
        enz_emb = get_nonconserved_embedding(enzyme, conservation_threshold, pooling)
        if enz_emb is None:
            skipped += 1
            continue
        if amine not in amine_fingerprints:
            skipped += 1
            continue
        
        features = np.concatenate([enz_emb, amine_fingerprints[amine]])
        X_list.append(features)
        y_list.append(int(row['active']))
        enzymes.append(enzyme)
        amines.append(amine)
    
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    
    return X, y, enzymes, amines, skipped


# Also build full protein embedding baseline for comparison
h5_full = DATA_DIR / "Seqs_list_total.h5"
full_embeddings = {}
with h5py.File(h5_full, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        full_embeddings[uniprot_id] = f[key][:]

print(f"Loaded {len(full_embeddings)} full protein embeddings")

# Build baseline feature matrix
X_base_list, y_base_list = [], []
enz_base, ami_base = [], []
for _, row in df_agg.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    if enzyme not in full_embeddings or amine not in amine_fingerprints:
        continue
    features = np.concatenate([full_embeddings[enzyme], amine_fingerprints[amine]])
    X_base_list.append(features)
    y_base_list.append(int(row['active']))
    enz_base.append(enzyme)
    ami_base.append(amine)

X_baseline = np.array(X_base_list, dtype=np.float32)
y_baseline = np.array(y_base_list, dtype=np.int32)

print(f"\nBaseline (full protein): {X_baseline.shape}")

# Build for each threshold
datasets = {'full_protein': (X_baseline, y_baseline, enz_base, ami_base)}

for name, thresh in THRESHOLDS.items():
    X, y, enzymes, amines, skipped = build_feature_matrix(thresh)
    datasets[name] = (X, y, enzymes, amines)
    print(f"{name} (< {thresh}): {X.shape}, skipped={skipped}")

## 5. Enzyme Hold-Out Split & Train Models

In [ ]:
def enzyme_holdout_split(X, y, enzymes, amines, test_size=0.2, val_size=0.2):
    """
    Split data by enzyme identity. Returns train/val/test splits.
    """
    enzymes_arr = np.array(enzymes)
    unique_enzymes = np.unique(enzymes_arr)
    
    # Compute enzyme activity profiles for stratification
    profiles = np.array([y[enzymes_arr == e].mean() for e in unique_enzymes])
    bins = pd.cut(profiles, bins=5, labels=False)
    
    # Split enzymes into train+val and test
    train_val_enz, test_enz = train_test_split(
        unique_enzymes, test_size=test_size,
        random_state=RANDOM_STATE, stratify=bins
    )
    
    # Further split train into train and val
    tv_profiles = np.array([y[enzymes_arr == e].mean() for e in train_val_enz])
    tv_bins = pd.cut(tv_profiles, bins=5, labels=False)
    
    train_enz, val_enz = train_test_split(
        train_val_enz, test_size=val_size,
        random_state=RANDOM_STATE, stratify=tv_bins
    )
    
    train_mask = np.isin(enzymes_arr, train_enz)
    val_mask = np.isin(enzymes_arr, val_enz)
    test_mask = np.isin(enzymes_arr, test_enz)
    
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
        'train_enz': train_enz, 'val_enz': val_enz, 'test_enz': test_enz,
        'test_enzymes_arr': enzymes_arr[test_mask],
        'test_amines_arr': np.array(amines)[test_mask],
    }


# Create splits for each dataset
splits = {}
for name, (X, y, enzymes, amines) in datasets.items():
    splits[name] = enzyme_holdout_split(X, y, enzymes, amines)
    s = splits[name]
    print(f"{name}: train={len(s['X_train'])}, val={len(s['X_val'])}, test={len(s['X_test'])}")
    print(f"  Train active: {s['y_train'].mean():.1%}, Test active: {s['y_test'].mean():.1%}")

In [ ]:
def train_and_evaluate(split_data, dataset_name):
    """
    Train RF, XGBoost, MLP on the given split and return metrics.
    """
    X_train = split_data['X_train']
    y_train = split_data['y_train']
    X_val = split_data['X_val']
    y_val = split_data['y_val']
    X_test = split_data['X_test']
    y_test = split_data['y_test']
    
    # Class weights
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    class_weight = {0: 1.0, 1: n_neg / n_pos}
    
    # Scale for MLP
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_val_sc = scaler.transform(X_val)
    X_test_sc = scaler.transform(X_test)
    
    results = []
    models = {}
    
    # --- Random Forest ---
    rf = RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_leaf=5,
        class_weight=class_weight, random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    models['RF'] = rf
    
    # --- XGBoost ---
    if HAS_XGB:
        xgb_model = xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            scale_pos_weight=n_neg / n_pos,
            reg_alpha=0.1, reg_lambda=1.0,
            random_state=RANDOM_STATE,
            early_stopping_rounds=20, eval_metric='logloss', n_jobs=-1
        )
        xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        models['XGBoost'] = xgb_model
    
    # --- MLP ---
    mlp = MLPClassifier(
        hidden_layer_sizes=(256, 128, 64), activation='relu',
        alpha=0.01, batch_size=64, learning_rate_init=0.001,
        max_iter=500, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=10, random_state=RANDOM_STATE, verbose=False
    )
    mlp.fit(X_train_sc, y_train)
    models['MLP'] = (mlp, scaler)
    
    # --- Evaluate all models ---
    for model_name, model in models.items():
        if model_name == 'MLP':
            mlp_model, sc = model
            y_pred = mlp_model.predict(X_test_sc)
            y_proba = mlp_model.predict_proba(X_test_sc)[:, 1]
        else:
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)[:, 1]
        
        # Also get train and val scores for overfitting check
        if model_name == 'MLP':
            train_acc = mlp_model.score(X_train_sc, y_train)
            val_acc = mlp_model.score(X_val_sc, y_val)
        else:
            train_acc = model.score(X_train, y_train)
            val_acc = model.score(X_val, y_val)
        
        metrics = {
            'dataset': dataset_name,
            'model': model_name,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'test_accuracy': accuracy_score(y_test, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, y_proba),
            'pr_auc': average_precision_score(y_test, y_proba),
        }
        results.append(metrics)
    
    return results, models

print("Training function ready.")

In [ ]:
%%time

# Train all models on all datasets
all_results = []
all_models = {}

for name in datasets.keys():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")
    
    results, models = train_and_evaluate(splits[name], name)
    all_results.extend(results)
    all_models[name] = models
    
    for r in results:
        print(f"  {r['model']:8s} - ROC-AUC: {r['roc_auc']:.3f}, PR-AUC: {r['pr_auc']:.3f}, F1: {r['f1']:.3f}")
        print(f"           Train: {r['train_acc']:.3f}, Val: {r['val_acc']:.3f}, Test: {r['test_accuracy']:.3f}")

df_all_results = pd.DataFrame(all_results)
print("\nDone!")

## 6. Results Comparison

In [ ]:
# Full results table
print("\n" + "="*80)
print("FULL RESULTS: Non-Conserved Residue Embeddings vs Full Protein")
print("="*80)

display_cols = ['dataset', 'model', 'train_acc', 'val_acc', 'test_accuracy', 
                'f1', 'roc_auc', 'pr_auc']
print(df_all_results[display_cols].to_string(index=False, float_format='%.3f'))

In [ ]:
# Overfitting check: train vs val vs test accuracy
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, model_name in enumerate(['RF', 'XGBoost', 'MLP']):
    ax = axes[idx]
    df_model = df_all_results[df_all_results['model'] == model_name]
    
    x = np.arange(len(df_model))
    width = 0.25
    
    ax.bar(x - width, df_model['train_acc'], width, label='Train', color='#3498db')
    ax.bar(x, df_model['val_acc'], width, label='Val', color='#2ecc71')
    ax.bar(x + width, df_model['test_accuracy'], width, label='Test', color='#e74c3c')
    
    ax.set_xlabel('Dataset', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_title(f'{model_name}: Train vs Val vs Test', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df_model['dataset'], rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(NONCONS_OUTPUT_DIR / 'overfitting_check.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {NONCONS_OUTPUT_DIR / 'overfitting_check.png'}")

In [ ]:
# Compare ROC-AUC and PR-AUC across datasets and models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

dataset_names = df_all_results['dataset'].unique()
model_names = df_all_results['model'].unique()
colors = {'RF': '#3498db', 'XGBoost': '#2ecc71', 'MLP': '#e74c3c'}

for ax, metric, title in zip(axes, ['roc_auc', 'pr_auc'], ['ROC-AUC', 'PR-AUC']):
    x = np.arange(len(dataset_names))
    width = 0.25
    
    for i, model in enumerate(model_names):
        vals = [df_all_results[(df_all_results['dataset'] == d) & 
                               (df_all_results['model'] == model)][metric].values[0]
                for d in dataset_names]
        offset = (i - 1) * width
        ax.bar(x + offset, vals, width, label=model, color=colors.get(model, 'gray'),
               edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Embedding Type', fontsize=12)
    ax.set_ylabel(title, fontsize=12)
    ax.set_title(f'{title} by Embedding Strategy', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(dataset_names, rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=10)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add 0.5 reference line for ROC-AUC (random)
    if metric == 'roc_auc':
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random')

plt.tight_layout()
plt.savefig(NONCONS_OUTPUT_DIR / 'embedding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {NONCONS_OUTPUT_DIR / 'embedding_comparison.png'}")

In [ ]:
# ROC curves for best model across datasets
best_model_name = df_all_results.loc[df_all_results['roc_auc'].idxmax(), 'model']
print(f"Best model overall: {best_model_name}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cmap = plt.cm.Set2
dataset_colors = {name: cmap(i) for i, name in enumerate(dataset_names)}

for ax, metric_func, curve_func, title in [
    (axes[0], roc_auc_score, roc_curve, 'ROC Curves'),
    (axes[1], average_precision_score, precision_recall_curve, 'PR Curves')
]:
    for name in dataset_names:
        s = splits[name]
        X_test = s['X_test']
        y_test = s['y_test']
        
        model = all_models[name].get(best_model_name)
        if model is None:
            continue
        
        if best_model_name == 'MLP':
            mlp_m, sc = model
            y_proba = mlp_m.predict_proba(sc.transform(X_test))[:, 1]
        else:
            y_proba = model.predict_proba(X_test)[:, 1]
        
        auc_val = metric_func(y_test, y_proba)
        
        if curve_func == roc_curve:
            fpr, tpr, _ = curve_func(y_test, y_proba)
            ax.plot(fpr, tpr, linewidth=2, color=dataset_colors[name],
                    label=f'{name} (AUC={auc_val:.3f})')
        else:
            prec, rec, _ = curve_func(y_test, y_proba)
            ax.plot(rec, prec, linewidth=2, color=dataset_colors[name],
                    label=f'{name} (AUC={auc_val:.3f})')
    
    if curve_func == roc_curve:
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
        ax.set_xlabel('FPR', fontsize=12)
        ax.set_ylabel('TPR', fontsize=12)
    else:
        ax.set_xlabel('Recall', fontsize=12)
        ax.set_ylabel('Precision', fontsize=12)
    
    ax.set_title(f'{best_model_name}: {title}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(NONCONS_OUTPUT_DIR / 'roc_pr_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {NONCONS_OUTPUT_DIR / 'roc_pr_curves_comparison.png'}")

## 7. Save Results

In [ ]:
# Save all results
df_all_results.to_csv(NONCONS_OUTPUT_DIR / 'nonconserved_model_results.csv', index=False)
print(f"Saved: {NONCONS_OUTPUT_DIR / 'nonconserved_model_results.csv'}")

# Save best models
best_dataset = df_all_results.loc[df_all_results['pr_auc'].idxmax(), 'dataset']
best_model_type = df_all_results.loc[df_all_results['pr_auc'].idxmax(), 'model']
print(f"\nBest overall: {best_model_type} with {best_dataset} embeddings")

best_model = all_models[best_dataset][best_model_type]
with open(MODEL_DIR / 'bsh_nonconserved_best_model.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'dataset': best_dataset,
        'model_type': best_model_type,
        'metrics': df_all_results[df_all_results['pr_auc'] == df_all_results['pr_auc'].max()].iloc[0].to_dict()
    }, f)
print(f"Saved: {MODEL_DIR / 'bsh_nonconserved_best_model.pkl'}")

In [ ]:
# Summary
print("="*70)
print("NON-CONSERVED RESIDUE MODEL - SUMMARY")
print("="*70)

print("\nComparison of embedding strategies (best model per strategy):")
print("-"*70)

for ds_name in dataset_names:
    df_ds = df_all_results[df_all_results['dataset'] == ds_name]
    best = df_ds.loc[df_ds['pr_auc'].idxmax()]
    print(f"\n  {ds_name}:")
    print(f"    Best model: {best['model']}")
    print(f"    ROC-AUC: {best['roc_auc']:.3f}  |  PR-AUC: {best['pr_auc']:.3f}  |  F1: {best['f1']:.3f}")
    print(f"    Train: {best['train_acc']:.3f}  |  Val: {best['val_acc']:.3f}  |  Test: {best['test_accuracy']:.3f}")

print("\n" + "="*70)
overall_best = df_all_results.loc[df_all_results['pr_auc'].idxmax()]
print(f"OVERALL BEST: {overall_best['model']} + {overall_best['dataset']}")
print(f"  ROC-AUC: {overall_best['roc_auc']:.3f}")
print(f"  PR-AUC: {overall_best['pr_auc']:.3f}")
print(f"  F1: {overall_best['f1']:.3f}")
print("="*70)

print(f"\nFiles saved to: {NONCONS_OUTPUT_DIR}")

## 8. Pooling Strategies & Conserved + Non-Conserved Combined

Test whether different ways of aggregating per-residue embeddings improve performance:

1. **Mean pooling** (baseline) - average across selected residues
2. **Max pooling** - take max per feature dimension (captures strongest signals)
3. **Mean + Max concat** - concatenate both (2048-dim enzyme vector)
4. **Conserved + Non-conserved** - separate feature blocks for conserved vs variable residues
5. **Conserved + Non-conserved + full** - all three blocks combined

The idea: conserved residues tell the model "this is a BSH" while non-conserved residues tell it "this is what makes THIS BSH unique." Providing both as separate blocks lets the model learn from both signals.

In [ ]:
def get_embedding_flexible(enzyme_id, conservation_threshold, pooling='mean'):
    """
    Extract per-residue embeddings at non-conserved positions with flexible pooling.
    
    pooling options: 'mean', 'max', 'mean_max' (concatenated)
    Returns pooled vector or None.
    """
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    
    variable_aln_positions = df_core[
        df_core['conservation_score'] < conservation_threshold
    ]['alignment_position'].values
    
    seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    
    if len(seq_positions) == 0:
        return None
    
    selected = embed[seq_positions]
    
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    elif pooling == 'mean_max':
        return np.concatenate([selected.mean(axis=0), selected.max(axis=0)])
    
    return selected.mean(axis=0)


def get_conserved_embedding(enzyme_id, conservation_threshold=0.95, pooling='mean'):
    """
    Extract per-residue embeddings at CONSERVED positions (conservation >= threshold).
    """
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    
    conserved_aln_positions = df_core[
        df_core['conservation_score'] >= conservation_threshold
    ]['alignment_position'].values
    
    seq_positions = []
    for aln_pos in conserved_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    
    if len(seq_positions) == 0:
        return None
    
    selected = embed[seq_positions]
    
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    elif pooling == 'mean_max':
        return np.concatenate([selected.mean(axis=0), selected.max(axis=0)])
    
    return selected.mean(axis=0)


# Verify
test_enz = list(overlap)[0]
print(f"Test enzyme: {test_enz}")
for pool in ['mean', 'max', 'mean_max']:
    emb = get_embedding_flexible(test_enz, 0.5, pooling=pool)
    print(f"  Non-conserved {pool}: shape={emb.shape}")

cons_emb = get_conserved_embedding(test_enz)
print(f"  Conserved mean: shape={cons_emb.shape}")

In [ ]:
# Build all feature matrix variants using the best threshold (unique, < 0.5)
BEST_THRESH = 0.5

strategies = {}

# --- Strategy 1: Mean pooling (already have this as 'unique') ---
strategies['noncons_mean'] = datasets['unique']

# --- Strategy 2: Max pooling ---
X_list, y_list, enz_list, ami_list = [], [], [], []
for _, row in df_agg.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    emb = get_embedding_flexible(enzyme, BEST_THRESH, pooling='max')
    if emb is None or amine not in amine_fingerprints:
        continue
    X_list.append(np.concatenate([emb, amine_fingerprints[amine]]))
    y_list.append(int(row['active']))
    enz_list.append(enzyme)
    ami_list.append(amine)
strategies['noncons_max'] = (np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32), enz_list, ami_list)
print(f"noncons_max: {strategies['noncons_max'][0].shape}")

# --- Strategy 3: Mean+Max concatenated ---
X_list, y_list, enz_list, ami_list = [], [], [], []
for _, row in df_agg.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    emb = get_embedding_flexible(enzyme, BEST_THRESH, pooling='mean_max')
    if emb is None or amine not in amine_fingerprints:
        continue
    X_list.append(np.concatenate([emb, amine_fingerprints[amine]]))
    y_list.append(int(row['active']))
    enz_list.append(enzyme)
    ami_list.append(amine)
strategies['noncons_mean_max'] = (np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32), enz_list, ami_list)
print(f"noncons_mean_max: {strategies['noncons_mean_max'][0].shape}")

# --- Strategy 4: Conserved (mean) + Non-conserved (mean) as separate blocks ---
X_list, y_list, enz_list, ami_list = [], [], [], []
for _, row in df_agg.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    noncons_emb = get_embedding_flexible(enzyme, BEST_THRESH, pooling='mean')
    cons_emb = get_conserved_embedding(enzyme, conservation_threshold=0.95, pooling='mean')
    if noncons_emb is None or cons_emb is None or amine not in amine_fingerprints:
        continue
    # [conserved_1024 | nonconserved_1024 | amine_1024]
    X_list.append(np.concatenate([cons_emb, noncons_emb, amine_fingerprints[amine]]))
    y_list.append(int(row['active']))
    enz_list.append(enzyme)
    ami_list.append(amine)
strategies['cons_plus_noncons'] = (np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32), enz_list, ami_list)
print(f"cons_plus_noncons: {strategies['cons_plus_noncons'][0].shape}")

# --- Strategy 5: Full protein + Non-conserved as separate blocks ---
X_list, y_list, enz_list, ami_list = [], [], [], []
for _, row in df_agg.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    noncons_emb = get_embedding_flexible(enzyme, BEST_THRESH, pooling='mean')
    if noncons_emb is None or enzyme not in full_embeddings or amine not in amine_fingerprints:
        continue
    full_emb = full_embeddings[enzyme]
    # [full_protein_1024 | nonconserved_1024 | amine_1024]
    X_list.append(np.concatenate([full_emb, noncons_emb, amine_fingerprints[amine]]))
    y_list.append(int(row['active']))
    enz_list.append(enzyme)
    ami_list.append(amine)
strategies['full_plus_noncons'] = (np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32), enz_list, ami_list)
print(f"full_plus_noncons: {strategies['full_plus_noncons'][0].shape}")

# Add baselines for comparison
strategies['full_protein'] = datasets['full_protein']

print(f"\n{len(strategies)} strategies ready to train.")

In [ ]:
%%time

# Train all strategies
strategy_results = []
strategy_models = {}

for name, (X, y, enzymes, amines) in strategies.items():
    print(f"\n{'='*50}")
    print(f"Strategy: {name} (features: {X.shape[1]})")
    print(f"{'='*50}")
    
    split = enzyme_holdout_split(X, y, enzymes, amines)
    results, models = train_and_evaluate(split, name)
    strategy_results.extend(results)
    strategy_models[name] = {'models': models, 'split': split}
    
    for r in results:
        print(f"  {r['model']:8s} | ROC: {r['roc_auc']:.3f} | PR: {r['pr_auc']:.3f} | F1: {r['f1']:.3f} | Train: {r['train_acc']:.3f} | Test: {r['test_accuracy']:.3f}")

df_strategy_results = pd.DataFrame(strategy_results)
print("\nDone!")

In [ ]:
# Visualize pooling strategy comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

strategy_names = df_strategy_results['dataset'].unique()
model_names_s = df_strategy_results['model'].unique()
colors_s = {'RF': '#3498db', 'XGBoost': '#2ecc71', 'MLP': '#e74c3c'}

for ax, metric, title in zip(axes, ['roc_auc', 'pr_auc', 'f1'], ['ROC-AUC', 'PR-AUC', 'F1']):
    x = np.arange(len(strategy_names))
    width = 0.25
    
    for i, model in enumerate(model_names_s):
        vals = [df_strategy_results[(df_strategy_results['dataset'] == d) & 
                                     (df_strategy_results['model'] == model)][metric].values[0]
                for d in strategy_names]
        offset = (i - 1) * width
        ax.bar(x + offset, vals, width, label=model, color=colors_s.get(model, 'gray'),
               edgecolor='black', linewidth=0.5)
    
    ax.set_ylabel(title, fontsize=12)
    ax.set_title(f'{title}: Pooling & Feature Strategy', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(strategy_names, rotation=40, ha='right', fontsize=9)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(NONCONS_OUTPUT_DIR / 'pooling_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {NONCONS_OUTPUT_DIR / 'pooling_strategy_comparison.png'}")

In [ ]:
# Summary table: best model per strategy
print("="*80)
print("POOLING & FEATURE STRATEGY COMPARISON")
print("="*80)
print(f"\n{'Strategy':<22} {'Model':<10} {'Features':>8} {'ROC-AUC':>9} {'PR-AUC':>9} {'F1':>7} {'Train':>7} {'Test':>7}")
print("-"*80)

for strat in strategy_names:
    df_s = df_strategy_results[df_strategy_results['dataset'] == strat]
    best = df_s.loc[df_s['pr_auc'].idxmax()]
    n_feat = strategies[strat][0].shape[1]
    print(f"{strat:<22} {best['model']:<10} {n_feat:>8} {best['roc_auc']:>9.3f} {best['pr_auc']:>9.3f} {best['f1']:>7.3f} {best['train_acc']:>7.3f} {best['test_accuracy']:>7.3f}")

print("\n" + "="*80)
overall = df_strategy_results.loc[df_strategy_results['pr_auc'].idxmax()]
print(f"BEST OVERALL: {overall['model']} + {overall['dataset']}")
print(f"  ROC-AUC: {overall['roc_auc']:.3f}  |  PR-AUC: {overall['pr_auc']:.3f}  |  F1: {overall['f1']:.3f}")
print("="*80)

# Save results
df_strategy_results.to_csv(NONCONS_OUTPUT_DIR / 'pooling_strategy_results.csv', index=False)
print(f"\nSaved: {NONCONS_OUTPUT_DIR / 'pooling_strategy_results.csv'}")

## 9. Multi-Split Overfitting Assessment

Run the top strategies across **10 different random enzyme hold-out splits** to:
1. Check if results are stable or dependent on one lucky split
2. Measure the true train-test gap (overfitting) with confidence intervals
3. Use the variance to guide regularization

In [ ]:
def enzyme_holdout_split_seed(X, y, enzymes, amines, seed, test_size=0.2, val_size=0.2):
    """Enzyme hold-out split with a specific random seed."""
    enzymes_arr = np.array(enzymes)
    unique_enzymes = np.unique(enzymes_arr)
    
    profiles = np.array([y[enzymes_arr == e].mean() for e in unique_enzymes])
    bins = pd.cut(profiles, bins=5, labels=False)
    
    train_val_enz, test_enz = train_test_split(
        unique_enzymes, test_size=test_size, random_state=seed, stratify=bins
    )
    
    tv_profiles = np.array([y[enzymes_arr == e].mean() for e in train_val_enz])
    tv_bins = pd.cut(tv_profiles, bins=5, labels=False)
    
    train_enz, val_enz = train_test_split(
        train_val_enz, test_size=val_size, random_state=seed, stratify=tv_bins
    )
    
    train_mask = np.isin(enzymes_arr, train_enz)
    val_mask = np.isin(enzymes_arr, val_enz)
    test_mask = np.isin(enzymes_arr, test_enz)
    
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
        'train_enz': train_enz, 'val_enz': val_enz, 'test_enz': test_enz,
    }


def train_evaluate_single(split_data, model_type='XGBoost'):
    """Train a single model and return metrics dict."""
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    
    if model_type == 'XGBoost':
        model = xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            scale_pos_weight=n_neg / n_pos,
            reg_alpha=0.1, reg_lambda=1.0,
            random_state=42, early_stopping_rounds=20,
            eval_metric='logloss', n_jobs=-1
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    elif model_type == 'RF':
        model = RandomForestClassifier(
            n_estimators=100, max_depth=20, min_samples_leaf=5,
            class_weight={0: 1.0, 1: n_neg/n_pos},
            random_state=42, n_jobs=-1
        )
        model.fit(X_train, y_train)
    elif model_type == 'MLP':
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        model = MLPClassifier(
            hidden_layer_sizes=(256, 128, 64), activation='relu',
            alpha=0.01, batch_size=64, learning_rate_init=0.001,
            max_iter=500, early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=10, random_state=42, verbose=False
        )
        model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    train_acc = model.score(X_train, y_train)
    val_acc = model.score(X_val, y_val)
    
    return {
        'train_acc': train_acc,
        'val_acc': val_acc,
        'test_acc': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
    }

print("Multi-split functions ready.")

In [ ]:
%%time

# Run 10 random splits on top 3 strategies + baseline
N_SPLITS = 10
SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]

top_strategies = {
    'noncons_mean': strategies['noncons_mean'],
    'noncons_max': strategies['noncons_max'],
    'cons_plus_noncons': strategies['cons_plus_noncons'],
    'full_protein': strategies['full_protein'],
}

multi_split_results = []

for strat_name, (X, y, enzymes, amines) in top_strategies.items():
    for model_type in ['XGBoost', 'RF', 'MLP']:
        for i, seed in enumerate(SEEDS):
            split = enzyme_holdout_split_seed(X, y, enzymes, amines, seed=seed)
            metrics = train_evaluate_single(split, model_type=model_type)
            metrics['strategy'] = strat_name
            metrics['model'] = model_type
            metrics['split_seed'] = seed
            metrics['split_idx'] = i
            multi_split_results.append(metrics)
        
        # Print progress
        last_results = [r for r in multi_split_results 
                       if r['strategy'] == strat_name and r['model'] == model_type]
        roc_vals = [r['roc_auc'] for r in last_results]
        print(f"{strat_name} + {model_type}: ROC-AUC = {np.mean(roc_vals):.3f} +/- {np.std(roc_vals):.3f}")

df_multi = pd.DataFrame(multi_split_results)
print(f"\nTotal experiments: {len(df_multi)}")
print("Done!")

In [ ]:
# Multi-split summary table
print("="*90)
print("MULTI-SPLIT RESULTS (mean +/- std over 10 random enzyme hold-out splits)")
print("="*90)
print(f"\n{'Strategy':<22} {'Model':<10} {'Train':>12} {'Val':>12} {'Test':>12} {'ROC-AUC':>14} {'PR-AUC':>14} {'F1':>12}")
print("-"*90)

summary_rows = []
for strat in top_strategies.keys():
    for model in ['XGBoost', 'RF', 'MLP']:
        mask = (df_multi['strategy'] == strat) & (df_multi['model'] == model)
        df_sub = df_multi[mask]
        
        row = {
            'strategy': strat, 'model': model,
            'train_mean': df_sub['train_acc'].mean(), 'train_std': df_sub['train_acc'].std(),
            'val_mean': df_sub['val_acc'].mean(), 'val_std': df_sub['val_acc'].std(),
            'test_mean': df_sub['test_acc'].mean(), 'test_std': df_sub['test_acc'].std(),
            'roc_mean': df_sub['roc_auc'].mean(), 'roc_std': df_sub['roc_auc'].std(),
            'pr_mean': df_sub['pr_auc'].mean(), 'pr_std': df_sub['pr_auc'].std(),
            'f1_mean': df_sub['f1'].mean(), 'f1_std': df_sub['f1'].std(),
            'gap': df_sub['train_acc'].mean() - df_sub['test_acc'].mean(),
        }
        summary_rows.append(row)
        
        print(f"{strat:<22} {model:<10} "
              f"{row['train_mean']:.3f}+/-{row['train_std']:.3f} "
              f"{row['val_mean']:.3f}+/-{row['val_std']:.3f} "
              f"{row['test_mean']:.3f}+/-{row['test_std']:.3f} "
              f"{row['roc_mean']:.3f}+/-{row['roc_std']:.3f} "
              f"{row['pr_mean']:.3f}+/-{row['pr_std']:.3f} "
              f"{row['f1_mean']:.3f}+/-{row['f1_std']:.3f}")

df_summary = pd.DataFrame(summary_rows)

print(f"\n{'='*90}")
print("OVERFITTING GAP (Train Acc - Test Acc):")
print("-"*50)
for _, r in df_summary.iterrows():
    bar = "#" * int(r['gap'] * 100)
    print(f"  {r['strategy']:<22} {r['model']:<10} gap={r['gap']:.3f}  {bar}")

In [ ]:
# Visualization: box plots across splits
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

metrics_to_plot = ['train_acc', 'test_acc', 'roc_auc', 'pr_auc', 'f1', 'recall']
titles = ['Train Accuracy', 'Test Accuracy', 'ROC-AUC', 'PR-AUC', 'F1', 'Recall']

for ax, metric, title in zip(axes.flat, metrics_to_plot, titles):
    data_to_plot = []
    labels = []
    
    for strat in top_strategies.keys():
        for model in ['XGBoost']:  # Focus on XGBoost as the consistently best model
            mask = (df_multi['strategy'] == strat) & (df_multi['model'] == model)
            data_to_plot.append(df_multi[mask][metric].values)
            labels.append(strat)
    
    bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
    
    colors_box = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax.set_title(f'{title} (XGBoost)', fontsize=12, fontweight='bold')
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Distribution Across 10 Random Splits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(NONCONS_OUTPUT_DIR / 'multi_split_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {NONCONS_OUTPUT_DIR / 'multi_split_boxplots.png'}")

In [ ]:
# Train-Test gap visualization
fig, ax = plt.subplots(figsize=(12, 6))

strat_labels = []
for strat in top_strategies.keys():
    for model in ['XGBoost', 'RF', 'MLP']:
        mask = (df_multi['strategy'] == strat) & (df_multi['model'] == model)
        df_sub = df_multi[mask]
        strat_labels.append(f"{strat}\n{model}")

x = np.arange(len(strat_labels))
train_means = df_summary['train_mean'].values
test_means = df_summary['test_mean'].values
train_stds = df_summary['train_std'].values
test_stds = df_summary['test_std'].values

ax.bar(x - 0.2, train_means, 0.35, yerr=train_stds, label='Train', 
       color='#3498db', alpha=0.8, capsize=3)
ax.bar(x + 0.2, test_means, 0.35, yerr=test_stds, label='Test', 
       color='#e74c3c', alpha=0.8, capsize=3)

# Add gap annotations
for i, (tr, te) in enumerate(zip(train_means, test_means)):
    gap = tr - te
    ax.annotate(f'{gap:.2f}', xy=(i, max(tr, te) + 0.02), ha='center', fontsize=8, color='gray')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Train vs Test Accuracy (mean +/- std across 10 splits)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(strat_labels, fontsize=8, ha='center')
ax.legend(fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(NONCONS_OUTPUT_DIR / 'multi_split_train_test_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {NONCONS_OUTPUT_DIR / 'multi_split_train_test_gap.png'}")

## 10. Regularized Models to Reduce Overfitting

Based on the multi-split analysis, retrain with stronger regularization:
- **XGBoost**: Lower max_depth, higher reg_alpha/lambda, lower learning rate, fewer estimators
- **RF**: Lower max_depth, higher min_samples_leaf
- **MLP**: Higher alpha (L2), smaller network

In [ ]:
def train_evaluate_regularized(split_data, model_type='XGBoost'):
    """Train with stronger regularization to reduce overfitting."""
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    
    if model_type == 'XGBoost':
        model = xgb.XGBClassifier(
            n_estimators=300, max_depth=3,       # shallower trees
            learning_rate=0.05,                   # slower learning
            scale_pos_weight=n_neg / n_pos,
            reg_alpha=1.0, reg_lambda=5.0,        # much stronger regularization
            subsample=0.7, colsample_bytree=0.7,  # feature/sample subsampling
            min_child_weight=5,                    # prevent small leaves
            random_state=42, early_stopping_rounds=30,
            eval_metric='logloss', n_jobs=-1
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    elif model_type == 'RF':
        model = RandomForestClassifier(
            n_estimators=200, max_depth=10,       # shallower trees
            min_samples_leaf=15,                   # larger leaves
            min_samples_split=10,
            max_features='sqrt',
            class_weight={0: 1.0, 1: n_neg/n_pos},
            random_state=42, n_jobs=-1
        )
        model.fit(X_train, y_train)
    elif model_type == 'MLP':
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        model = MLPClassifier(
            hidden_layer_sizes=(128, 64),          # smaller network
            activation='relu',
            alpha=0.1,                             # 10x stronger L2
            batch_size=64, learning_rate_init=0.0005,  # slower learning
            max_iter=500, early_stopping=True, validation_fraction=0.15,
            n_iter_no_change=15, random_state=42, verbose=False
        )
        model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    return {
        'train_acc': model.score(X_train, y_train),
        'val_acc': model.score(X_val, y_val),
        'test_acc': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
    }

print("Regularized training function ready.")

In [ ]:
%%time

# Run regularized models across same 10 splits
reg_results = []

for strat_name, (X, y, enzymes, amines) in top_strategies.items():
    for model_type in ['XGBoost', 'RF', 'MLP']:
        for i, seed in enumerate(SEEDS):
            split = enzyme_holdout_split_seed(X, y, enzymes, amines, seed=seed)
            metrics = train_evaluate_regularized(split, model_type=model_type)
            metrics['strategy'] = strat_name
            metrics['model'] = model_type
            metrics['split_seed'] = seed
            metrics['split_idx'] = i
            reg_results.append(metrics)
        
        last = [r for r in reg_results if r['strategy'] == strat_name and r['model'] == model_type]
        roc_vals = [r['roc_auc'] for r in last]
        gap_vals = [r['train_acc'] - r['test_acc'] for r in last]
        print(f"{strat_name} + {model_type}: ROC-AUC = {np.mean(roc_vals):.3f}+/-{np.std(roc_vals):.3f}, gap = {np.mean(gap_vals):.3f}")

df_reg = pd.DataFrame(reg_results)
print("\nDone!")

In [ ]:
# Compare original vs regularized: focus on XGBoost
print("="*95)
print("ORIGINAL vs REGULARIZED XGBoost (mean +/- std over 10 splits)")
print("="*95)
print(f"\n{'Strategy':<22} {'Version':<12} {'Train':>10} {'Test':>10} {'Gap':>7} {'ROC-AUC':>14} {'PR-AUC':>14} {'F1':>12}")
print("-"*95)

comparison_rows = []
for strat in top_strategies.keys():
    for label, df_source in [('Original', df_multi), ('Regularized', df_reg)]:
        mask = (df_source['strategy'] == strat) & (df_source['model'] == 'XGBoost')
        df_sub = df_source[mask]
        
        row = {
            'strategy': strat, 'version': label,
            'train': df_sub['train_acc'].mean(), 'train_std': df_sub['train_acc'].std(),
            'test': df_sub['test_acc'].mean(), 'test_std': df_sub['test_acc'].std(),
            'gap': df_sub['train_acc'].mean() - df_sub['test_acc'].mean(),
            'roc': df_sub['roc_auc'].mean(), 'roc_std': df_sub['roc_auc'].std(),
            'pr': df_sub['pr_auc'].mean(), 'pr_std': df_sub['pr_auc'].std(),
            'f1': df_sub['f1'].mean(), 'f1_std': df_sub['f1'].std(),
        }
        comparison_rows.append(row)
        
        print(f"{strat:<22} {label:<12} "
              f"{row['train']:.3f}     {row['test']:.3f}     {row['gap']:.3f} "
              f"{row['roc']:.3f}+/-{row['roc_std']:.3f} "
              f"{row['pr']:.3f}+/-{row['pr_std']:.3f} "
              f"{row['f1']:.3f}+/-{row['f1_std']:.3f}")
    print()

df_comparison = pd.DataFrame(comparison_rows)

In [ ]:
# Visual comparison: original vs regularized
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, metric, title in zip(axes, ['roc', 'pr', 'gap'], ['ROC-AUC', 'PR-AUC', 'Overfitting Gap']):
    strats = list(top_strategies.keys())
    x = np.arange(len(strats))
    width = 0.35
    
    orig_vals = [df_comparison[(df_comparison['strategy'] == s) & (df_comparison['version'] == 'Original')][metric].values[0] for s in strats]
    reg_vals = [df_comparison[(df_comparison['strategy'] == s) & (df_comparison['version'] == 'Regularized')][metric].values[0] for s in strats]
    
    ax.bar(x - width/2, orig_vals, width, label='Original', color='#e74c3c', alpha=0.8)
    ax.bar(x + width/2, reg_vals, width, label='Regularized', color='#2ecc71', alpha=0.8)
    
    ax.set_ylabel(title, fontsize=12)
    ax.set_title(f'{title}: Original vs Regularized (XGBoost)', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(strats, rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    if metric == 'gap':
        ax.set_ylim(0, 0.35)
    else:
        ax.set_ylim(0.4, 0.9)

plt.tight_layout()
plt.savefig(NONCONS_OUTPUT_DIR / 'original_vs_regularized.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {NONCONS_OUTPUT_DIR / 'original_vs_regularized.png'}")

In [ ]:
# Save all multi-split results
df_multi.to_csv(NONCONS_OUTPUT_DIR / 'multi_split_original.csv', index=False)
df_reg.to_csv(NONCONS_OUTPUT_DIR / 'multi_split_regularized.csv', index=False)
df_comparison.to_csv(NONCONS_OUTPUT_DIR / 'original_vs_regularized_summary.csv', index=False)

print("Saved:")
print(f"  {NONCONS_OUTPUT_DIR / 'multi_split_original.csv'}")
print(f"  {NONCONS_OUTPUT_DIR / 'multi_split_regularized.csv'}")
print(f"  {NONCONS_OUTPUT_DIR / 'original_vs_regularized_summary.csv'}")

# Final summary
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print("\nOriginal vs Regularized XGBoost (best strategy per version):")

for label, df_src in [('Original', df_comparison[df_comparison['version'] == 'Original']),
                       ('Regularized', df_comparison[df_comparison['version'] == 'Regularized'])]:
    best = df_src.loc[df_src['pr'].idxmax()]
    print(f"\n  {label}:")
    print(f"    Strategy: {best['strategy']}")
    print(f"    ROC-AUC: {best['roc']:.3f}")
    print(f"    PR-AUC:  {best['pr']:.3f}")
    print(f"    F1:      {best['f1']:.3f}")
    print(f"    Train-Test Gap: {best['gap']:.3f}")

## 11. Signal Peptide Analysis

**Hypothesis:** Some BSH enzymes appear inactive because their signal peptides
are not properly cleaved by E. coli signal peptidases. If we "trim" the signal
peptide from the embeddings and predict with just the mature protein, would the
model predict activity?

**Approach:**
- Use existing per-residue embeddings but exclude signal peptide residue positions
- Pool only the mature protein residues (after signal peptide)
- Compare model predictions: full sequence vs trimmed sequence
- Focus on enzymes that are currently inactive — does trimming change their prediction?

In [ ]:
# Load signal peptide data
from Bio import SeqIO as SeqIO_sp

# Full sequences with signal peptides
sigpep_full = {}
for rec in SeqIO_sp.parse(DATA_DIR / "SignalPeptides.txt", 'fasta'):
    uid = rec.id.split('_')[-1]
    sigpep_full[uid] = str(rec.seq)

# Signal peptide sequences only (to get lengths)
sigpep_only = {}
for rec in SeqIO_sp.parse(DATA_DIR / "sigpep_seqs.txt", 'fasta'):
    uid = rec.id.split('_')[-1]
    sigpep_only[uid] = str(rec.seq)

# Trimmed sequences (mature protein without signal peptide)
trimmed_seqs = {}
for rec in SeqIO_sp.parse(DATA_DIR / "proteins without signal peptides trimmed.txt", 'fasta'):
    uid = rec.id.split('_')[-1]
    trimmed_seqs[uid] = str(rec.seq)

print(f"Enzymes with signal peptides: {len(sigpep_only)}")

# Signal peptide lengths
sp_lengths = {uid: len(seq) for uid, seq in sigpep_only.items()}
print(f"\nSignal peptide length distribution:")
lengths = list(sp_lengths.values())
print(f"  Min: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.0f}, Median: {np.median(lengths):.0f}")

# Classify by phylum (Bacteroidota vs Bacillota based on signal peptide length)
# Long signal peptides are more common in Bacillota
print(f"\n  <= 30 aa: {sum(1 for l in lengths if l <= 30)} enzymes")
print(f"  31-60 aa: {sum(1 for l in lengths if 31 <= l <= 60)} enzymes")
print(f"  > 60 aa:  {sum(1 for l in lengths if l > 60)} enzymes")

In [ ]:
def get_trimmed_embedding(enzyme_id, sp_length, conservation_threshold=0.5, pooling='mean'):
    """
    Get enzyme embedding using only mature protein residues (after signal peptide).
    Uses existing per-residue embeddings but skips the first sp_length positions.
    """
    if enzyme_id not in per_residue_embeddings:
        return None
    
    embed = per_residue_embeddings[enzyme_id]  # (seq_len, 1024)
    
    # Skip signal peptide residues
    mature_embed = embed[sp_length:]
    
    if len(mature_embed) == 0:
        return None
    
    if pooling == 'mean':
        return mature_embed.mean(axis=0)
    elif pooling == 'max':
        return mature_embed.max(axis=0)
    
    return mature_embed.mean(axis=0)


def get_full_embedding(enzyme_id, pooling='mean'):
    """Get embedding using all residues (including signal peptide)."""
    if enzyme_id not in per_residue_embeddings:
        return None
    
    embed = per_residue_embeddings[enzyme_id]
    
    if pooling == 'mean':
        return embed.mean(axis=0)
    elif pooling == 'max':
        return embed.max(axis=0)
    
    return embed.mean(axis=0)


# Get activity data for signal peptide enzymes
sigpep_enzymes = set(sigpep_only.keys())
df_sigpep_activity = df_agg[df_agg['Enzyme'].isin(sigpep_enzymes)].copy()

# Compute per-enzyme activity summary
enzyme_activity = df_sigpep_activity.groupby('Enzyme').agg(
    n_active=('active', 'sum'),
    n_total=('active', 'count'),
).reset_index()
enzyme_activity['pct_active'] = enzyme_activity['n_active'] / enzyme_activity['n_total']
enzyme_activity['sp_length'] = enzyme_activity['Enzyme'].map(sp_lengths)
enzyme_activity['is_inactive'] = enzyme_activity['pct_active'] == 0

print(f"Signal peptide enzymes in activity data: {len(enzyme_activity)}")
print(f"  Active (any): {(~enzyme_activity['is_inactive']).sum()}")
print(f"  Completely inactive: {enzyme_activity['is_inactive'].sum()}")

# Show inactive ones with signal peptide length
print(f"\nInactive enzymes with signal peptide lengths:")
inactive = enzyme_activity[enzyme_activity['is_inactive']].sort_values('sp_length', ascending=False)
for _, row in inactive.iterrows():
    print(f"  {row['Enzyme']:20s} SP length: {row['sp_length']:3d} aa")

In [ ]:
# Train a model on NON-signal-peptide enzymes, then predict on signal peptide enzymes
# This way the model learns from "clean" data and we test on the signal peptide cases

# Build training data from enzymes WITHOUT signal peptides
non_sp_enzymes = set(df_agg['Enzyme'].unique()) - sigpep_enzymes

X_train_sp, y_train_sp = [], []
enz_train_sp, ami_train_sp = [], []

for _, row in df_agg.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    if enzyme not in non_sp_enzymes:
        continue
    emb = get_nonconserved_embedding(enzyme, 0.5, pooling='mean')
    if emb is None or amine not in amine_fingerprints:
        continue
    X_train_sp.append(np.concatenate([emb, amine_fingerprints[amine]]))
    y_train_sp.append(int(row['active']))
    enz_train_sp.append(enzyme)
    ami_train_sp.append(amine)

X_train_sp = np.array(X_train_sp, dtype=np.float32)
y_train_sp = np.array(y_train_sp, dtype=np.int32)

print(f"Training data (non-signal-peptide enzymes): {X_train_sp.shape}")
print(f"  Active: {y_train_sp.sum()}/{len(y_train_sp)} ({y_train_sp.mean():.1%})")

# Train XGBoost on this
n_neg = (y_train_sp == 0).sum()
n_pos = (y_train_sp == 1).sum()

xgb_sp = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=n_neg / n_pos,
    reg_alpha=0.5, reg_lambda=3.0,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='logloss', n_jobs=-1
)
xgb_sp.fit(X_train_sp, y_train_sp, verbose=False)
print(f"Train accuracy: {xgb_sp.score(X_train_sp, y_train_sp):.3f}")

In [ ]:
# Predict on signal peptide enzymes: FULL sequence vs TRIMMED (no signal peptide)
sp_predictions = []

for _, row in df_sigpep_activity.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    actual_active = int(row['active'])
    
    if enzyme not in sp_lengths or amine not in amine_fingerprints:
        continue
    
    sp_len = sp_lengths[enzyme]
    fp = amine_fingerprints[amine]
    
    # Full sequence embedding (non-conserved positions, includes signal peptide influence)
    full_emb = get_nonconserved_embedding(enzyme, 0.5, pooling='mean')
    
    # Trimmed embedding (skip signal peptide residues, then pool)
    trimmed_emb = get_trimmed_embedding(enzyme, sp_len, pooling='mean')
    
    if full_emb is None or trimmed_emb is None:
        continue
    
    X_full = np.concatenate([full_emb, fp]).reshape(1, -1)
    X_trimmed = np.concatenate([trimmed_emb, fp]).reshape(1, -1)
    
    pred_full = xgb_sp.predict(X_full)[0]
    proba_full = xgb_sp.predict_proba(X_full)[0, 1]
    pred_trimmed = xgb_sp.predict(X_trimmed)[0]
    proba_trimmed = xgb_sp.predict_proba(X_trimmed)[0, 1]
    
    sp_predictions.append({
        'enzyme': enzyme,
        'amine': amine,
        'actual_active': actual_active,
        'sp_length': sp_len,
        'pred_full': pred_full,
        'proba_full': proba_full,
        'pred_trimmed': pred_trimmed,
        'proba_trimmed': proba_trimmed,
        'proba_change': proba_trimmed - proba_full,
        'flipped': pred_full != pred_trimmed,
    })

df_sp_pred = pd.DataFrame(sp_predictions)
print(f"Signal peptide predictions: {len(df_sp_pred)} enzyme-amine pairs")
print(f"Unique enzymes: {df_sp_pred['enzyme'].nunique()}")

In [ ]:
# Key analysis: for INACTIVE enzymes, does trimming flip predictions?
df_inactive_sp = df_sp_pred[df_sp_pred['actual_active'] == 0].copy()

print("="*80)
print("SIGNAL PEPTIDE ANALYSIS: Inactive enzyme-amine pairs")
print("="*80)

print(f"\nTotal inactive enzyme-amine pairs with signal peptides: {len(df_inactive_sp)}")
print(f"  Predicted active (full sequence): {(df_inactive_sp['pred_full'] == 1).sum()}")
print(f"  Predicted active (trimmed):       {(df_inactive_sp['pred_trimmed'] == 1).sum()}")
print(f"  Flipped to active after trimming: {((df_inactive_sp['pred_full'] == 0) & (df_inactive_sp['pred_trimmed'] == 1)).sum()}")

# Per-enzyme summary
print(f"\n{'='*80}")
print("PER-ENZYME SUMMARY (inactive enzymes with signal peptides)")
print(f"{'='*80}")
print(f"\n{'Enzyme':<20} {'SP len':>6} {'Amines':>7} {'Pred active':>12} {'Pred active':>12} {'Avg prob':>10} {'Avg prob':>10}")
print(f"{'':20} {'':>6} {'':>7} {'(full)':>12} {'(trimmed)':>12} {'(full)':>10} {'(trimmed)':>10}")
print("-"*80)

inactive_enzymes = enzyme_activity[enzyme_activity['is_inactive']]['Enzyme'].values

for enz in sorted(inactive_enzymes, key=lambda e: sp_lengths.get(e, 0), reverse=True):
    df_e = df_inactive_sp[df_inactive_sp['enzyme'] == enz]
    if len(df_e) == 0:
        continue
    sp_len = sp_lengths.get(enz, 0)
    n_amines = len(df_e)
    n_pred_full = (df_e['pred_full'] == 1).sum()
    n_pred_trim = (df_e['pred_trimmed'] == 1).sum()
    avg_prob_full = df_e['proba_full'].mean()
    avg_prob_trim = df_e['proba_trimmed'].mean()
    
    marker = " <-- GAINS" if n_pred_trim > n_pred_full else ""
    print(f"{enz:<20} {sp_len:>6} {n_amines:>7} {n_pred_full:>12} {n_pred_trim:>12} {avg_prob_full:>10.3f} {avg_prob_trim:>10.3f}{marker}")